pipeline lakeflow

In [0]:
from pyspark import pipelines as dp

from pyspark.sql.functions import *


---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:28
     27 try:
---> 28     from dlt import *
     29 except ImportError:

ModuleNotFoundError: No module named 'dlt'

During handling of the above exception, another exception occurred:

TypeError                                 Traceback (most recent call last)
File <command-6397784329315950>, line 1
----> 1 from pyspark import pipelines as dp
      3 from pyspark.sql.functions import col

File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:30
     28         from dlt import *
     29     except ImportError:
---> 30         raise PySparkException(errorClass="PIPELINES_NOT_SUPPORTED")
     31 # END: EDGE

File /databricks/python/lib/python3.12/site-packages/pyspark/errors/exceptions/base.py:51, in PySparkException.__init__(

In [0]:
@dlt.table(
    name="DimProducts_stage"
)
def dimProductsStage():
    df = (
        spark.readStream
        .table("databrick_cata.silver.products_silver")
        .filter(col("product_id").isNotNull())
        .dropDuplicates(["product_id"])
        .withColumn("update_date", current_timestamp())
    )

    return df

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6397784329315942>, line 1
----> 1 import dlt
      3 @dlt.table(
      4     name="DimProducts"
      5 )
      6 def dimProducts():
      7     return (
      8         spark.read
      9         .table("databrick_cata.silver.products_silver")
     10     )

ModuleNotFoundError: No module named 'dlt'

In [0]:
@dlt.view(
    name="DimProducts_view"
)
def dimProductsView():
    df = dlt.read_stream("DimProducts_stage")

    return df

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6397784329315947>, line 1
----> 1 @dlt.view
      2 def  DimProducts_view(): 
      3     df = spark.readStream.table("Live.DimProducts_stage")
      4     return df

NameError: name 'dlt' is not defined

In [0]:
dlt.create_streaming_table(
    name="DimProducts"
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6397784329315948>, line 1
----> 1 dlt.create_streaming_table("DimProducts")

NameError: name 'dlt' is not defined

In [0]:
dlt.apply_changes(
    target="DimProducts",
    source="DimProducts_view",
    keys=["product_id"],
    sequence_by=col("update_date"),
    stored_as_scd_type=1
)